# Fund the Future - Financial Model

Model projected long-term savings for Alice, Bob, Charlie, and Diana.
Each person creates a savings account with regular deposits and investment returns.

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import os

# Determine data path
data_paths = [
    '/workspace/data/',
    os.path.join(os.path.dirname(os.path.abspath('.')), 'environment', 'data'),
    '../environment/data/',
    'data/',
]
DATA_DIR = None
for p in data_paths:
    if os.path.isdir(p):
        DATA_DIR = p
        break
if DATA_DIR is None:
    DATA_DIR = '/workspace/data/'
print(f'Using data directory: {DATA_DIR}')

In [ ]:
# Read the Excel file to extract assumptions
# The file contains assumptions as text within sentences
import glob
xlsx_files = glob.glob(os.path.join(DATA_DIR, '*.xlsx'))
print(f'Found Excel files: {xlsx_files}')

if xlsx_files:
    wb = openpyxl.load_workbook(xlsx_files[0], data_only=True)
    print(f'Sheet names: {wb.sheetnames}')
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        print(f'\n=== Sheet: {sheet_name} ===')
        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column, values_only=False):
            vals = [(c.coordinate, c.value) for c in row if c.value is not None]
            if vals:
                print(vals)

## Key Assumptions

From the introduction and Excel file:

| Key Assumptions | Alice | Bob | Charlie | Diana |
|---|---|---|---|---|
| Initial Deposit (31 Dec 2016) | $12,820 | $0 | $40,000 | $5,000 |
| Annual Salary 2017 | $50,000 | $35,698 | $61,500 | $95,000 |
| Salary Growth | 3% p.a. | 2.2% 2018-2026, 2.8% 2027+ | $5,500 p.a. | 0% to 2025, 5% 2026-2040, 1.5% 2041+ |
| Core Deposits | 9% salary each 31 Dec | 3% salary each quarter | 7% salary each 31 Dec | 4% salary each 31 Dec |
| Additional Deposits | None | $20,000 on 30 Jun 2028 | $1,000 each quarter | $5,000 on 31 Dec 2021, every 5 yrs |
| Account Closes | 31 Dec 2044 | 31 Dec 2057 | 31 Dec 2039 | 31 Dec 2060 |
| Investment Returns | 4% p.a. annually | 1.10% p.q. quarterly | 0.90% p.q. quarterly | 8% p.a. 2017-2038, 5% 2039+ |

In [ ]:
# ============================================================
# ALICE MODEL - Annual compounding
# ============================================================
# Initial deposit: $12,820 at 31 Dec 2016
# Salary 2017: $50,000, grows 3% per year from 2018
# Core deposits: 9% of annual salary every 31 Dec (beginning 2017)
# Additional deposits: None
# Account closes: 31 Dec 2044
# Investment returns: 4% p.a. compounded annually each 31 Dec
#
# Order: closing = opening_balance * (1 + rate) + deposits
# (Investment returns calculated on opening balance of compounding period)

alice_balance = 12820.0
alice_salary = 50000.0
alice_balances = {2016: alice_balance}  # Balance at 31 Dec of each year
alice_core_deposits_total = 0.0
alice_inv_returns_total = 0.0

for year in range(2017, 2045):  # 2017 to 2044 inclusive
    # Salary for this year (grows each 1 January beginning 2018)
    if year == 2017:
        current_salary = 50000.0
    else:
        alice_salary = alice_salary * 1.03
        current_salary = alice_salary
    
    # Opening balance = previous closing balance
    opening = alice_balance
    
    # Investment return on opening balance (4% p.a.)
    inv_return = opening * 0.04
    alice_inv_returns_total += inv_return
    
    # Core deposit: 9% of annual salary
    core_deposit = current_salary * 0.09
    alice_core_deposits_total += core_deposit
    
    # Closing balance
    alice_balance = opening + inv_return + core_deposit
    alice_balances[year] = alice_balance

print(f"Alice balance at 31 Dec 2024: ${alice_balances[2024]:,.2f}")
print(f"Alice total core deposits: ${alice_core_deposits_total:,.2f}")
print(f"Alice final balance (31 Dec 2044): ${alice_balances[2044]:,.2f}")

In [ ]:
# ============================================================
# BOB MODEL - Quarterly compounding
# ============================================================
# Initial deposit: $0 at 31 Dec 2016
# Salary 2017: $35,698
# Salary growth: 2.2% per year from 2018 to 2026, 2.8% from 2027 onwards
# Core deposits: 3% of annual salary every quarter end
# Additional deposits: $20,000 on 30 June 2028
# Account closes: 31 Dec 2057
# Investment returns: 1.10% p.q. compounded each quarter end

bob_balance = 0.0
bob_salary = 35698.0
bob_balances = {2016: bob_balance}  # Balance at 31 Dec of each year
bob_inv_returns_by_year = {}  # Investment returns per year
bob_inv_returns_total = 0.0

for year in range(2017, 2058):  # 2017 to 2057 inclusive
    # Salary for this year (grows each 1 January beginning 2018)
    if year == 2017:
        current_salary = 35698.0
    else:
        if year <= 2026:
            bob_salary = bob_salary * 1.022
        else:
            bob_salary = bob_salary * 1.028
        current_salary = bob_salary
    
    yearly_inv_returns = 0.0
    
    # 4 quarters per year: Q1 (31 Mar), Q2 (30 Jun), Q3 (30 Sep), Q4 (31 Dec)
    for q in range(1, 5):
        opening = bob_balance
        
        # Investment return: 1.10% per quarter on opening balance
        inv_return = opening * 0.011
        yearly_inv_returns += inv_return
        
        # Core deposit: 3% of annual salary each quarter
        core_deposit = current_salary * 0.03
        
        # Additional deposit: $20,000 on 30 June 2028 (Q2)
        additional = 0.0
        if year == 2028 and q == 2:
            additional = 20000.0
        
        bob_balance = opening + inv_return + core_deposit + additional
    
    bob_balances[year] = bob_balance
    bob_inv_returns_by_year[year] = yearly_inv_returns
    bob_inv_returns_total += yearly_inv_returns

# Bob investment returns 2019-2023 inclusive
bob_inv_2019_2023 = sum(bob_inv_returns_by_year[y] for y in range(2019, 2024))
print(f"Bob investment returns 2019-2023: ${bob_inv_2019_2023:,.2f}")

# Year Bob first exceeds $200,000
bob_first_200k = None
for year in sorted(bob_balances.keys()):
    if bob_balances[year] > 200000:
        bob_first_200k = year
        break
print(f"Bob first exceeds $200,000 in: {bob_first_200k} (balance: ${bob_balances[bob_first_200k]:,.2f})")
print(f"Bob final balance (31 Dec 2057): ${bob_balances[2057]:,.2f}")

In [ ]:
# ============================================================
# CHARLIE MODEL - Quarterly compounding
# ============================================================
# Initial deposit: $40,000 at 31 Dec 2016
# Salary 2017: $61,500, grows by $5,500 per annum
# Core deposits: 7% of annual salary every 31 Dec (beginning 2017)
# Additional deposits: $1,000 every quarter end, beginning in 2017
# Account closes: 31 Dec 2039
# Investment returns: 0.90% p.q. compounded each quarter end

def model_charlie(close_year=2039):
    """Model Charlie's account up to close_year."""
    balance = 40000.0
    salary = 61500.0
    balances = {2016: balance}
    
    for year in range(2017, close_year + 1):
        # Salary for this year (grows by $5,500 each 1 January beginning 2018)
        if year == 2017:
            current_salary = 61500.0
        else:
            salary = salary + 5500.0
            current_salary = salary
        
        # 4 quarters per year
        for q in range(1, 5):
            opening = balance
            
            # Investment return: 0.90% per quarter on opening balance
            inv_return = opening * 0.009
            
            # Additional deposit: $1,000 every quarter end
            additional = 1000.0
            
            # Core deposit: 7% of annual salary on 31 Dec only (Q4)
            core_deposit = 0.0
            if q == 4:
                core_deposit = current_salary * 0.07
            
            balance = opening + inv_return + additional + core_deposit
        
        balances[year] = balance
    
    return balances, balance

charlie_balances, charlie_final = model_charlie(2039)
print(f"Charlie final balance (31 Dec 2039): ${charlie_final:,.2f}")

# For Q10: extend Charlie indefinitely
charlie_ext_balances, _ = model_charlie(2100)
charlie_first_1200k = None
for year in sorted(charlie_ext_balances.keys()):
    if charlie_ext_balances[year] > 1200000:
        charlie_first_1200k = year
        break
print(f"Charlie first exceeds $1,200,000 in: {charlie_first_1200k} (balance: ${charlie_ext_balances[charlie_first_1200k]:,.2f})")

In [ ]:
# ============================================================
# DIANA MODEL - Annual compounding
# ============================================================
# Initial deposit: $5,000 at 31 Dec 2016
# Salary 2017: $95,000
# Salary growth: 0% up to and including 2025, 5% from 2026 to 2040, 1.5% from 2041 onwards
# Core deposits: 4% of annual salary every 31 Dec (beginning 2017)
# Additional deposits: $5,000 on 31 Dec 2021 and each 5-year anniversary
# Account closes: 31 Dec 2060
# Investment returns: 8% p.a. from 2017 to 2038, 5% p.a. from 2039 onwards
#   Compounded each 31 Dec

diana_balance = 5000.0
diana_salary = 95000.0
diana_balances = {2016: diana_balance}
diana_salaries = {}
diana_inv_returns_total = 0.0

for year in range(2017, 2061):  # 2017 to 2060 inclusive
    # Salary for this year (grows each 1 January beginning 2018)
    if year == 2017:
        current_salary = 95000.0
    else:
        # Growth applied at start of year
        # 0% up to and including 2025 means no growth for 2018, 2019, ..., 2025
        # 5% from 2026 to 2040
        # 1.5% from 2041 onwards
        if year <= 2025:
            diana_salary = diana_salary * 1.00  # 0% growth
        elif year <= 2040:
            diana_salary = diana_salary * 1.05  # 5% growth
        else:
            diana_salary = diana_salary * 1.015  # 1.5% growth
        current_salary = diana_salary
    
    diana_salaries[year] = current_salary
    
    # Opening balance
    opening = diana_balance
    
    # Investment return
    if year <= 2038:
        inv_return = opening * 0.08
    else:
        inv_return = opening * 0.05
    diana_inv_returns_total += inv_return
    
    # Core deposit: 4% of annual salary
    core_deposit = current_salary * 0.04
    
    # Additional deposits: $5,000 on 31 Dec 2021 and each 5-year anniversary
    additional = 0.0
    if year >= 2021 and (year - 2021) % 5 == 0:
        additional = 5000.0
    
    diana_balance = opening + inv_return + core_deposit + additional
    diana_balances[year] = diana_balance

print(f"Diana salary in 2050: ${diana_salaries[2050]:,.2f}")
print(f"Diana total investment returns: ${diana_inv_returns_total:,.2f}")
print(f"Diana final balance (31 Dec 2060): ${diana_balances[2060]:,.2f}")

In [ ]:
# ============================================================
# ANSWER ALL QUESTIONS
# ============================================================

# Helper function: map numeric value to multiple choice letter
def match_mc(value, base_value):
    """Map a rounded value to multiple choice A-I starting from base_value."""
    rounded = round(value)
    offset = rounded - base_value
    if 0 <= offset <= 8:
        return chr(65 + offset)  # A=0, B=1, ..., I=8
    return f"OUT_OF_RANGE({rounded})"

# Q1: Alice's balance at 31 Dec 2024
# Options: A=$63,351 ... I=$63,359 (step 1)
q1 = match_mc(alice_balances[2024], 63351)
print(f"Q1: Alice balance 31 Dec 2024 = ${alice_balances[2024]:,.2f} -> {q1}")

# Q2: Total core deposits into Alice's account
# Options: A=$193,186 ... I=$193,194
q2 = match_mc(alice_core_deposits_total, 193186)
print(f"Q2: Alice total core deposits = ${alice_core_deposits_total:,.2f} -> {q2}")

# Q3: Bob's total investment returns 2019-2023 inclusive
# Options: A=$4,759 ... I=$4,767
q3 = match_mc(bob_inv_2019_2023, 4759)
print(f"Q3: Bob inv returns 2019-2023 = ${bob_inv_2019_2023:,.2f} -> {q3}")

# Q4: Year Bob's account first exceeds $200,000
# Options: A=2029, B=2030, ..., I=2037
q4 = match_mc(bob_first_200k, 2029)
print(f"Q4: Bob first > $200k in {bob_first_200k} -> {q4}")

# Q5: Charlie's final closing balance
# Options: A=$512,035 ... I=$512,043
q5 = match_mc(charlie_final, 512035)
print(f"Q5: Charlie final = ${charlie_final:,.2f} -> {q5}")

# Q6: Diana's salary in 2050
# Options: A=$229,203 ... I=$229,211
q6 = match_mc(diana_salaries[2050], 229203)
print(f"Q6: Diana salary 2050 = ${diana_salaries[2050]:,.2f} -> {q6}")

# Q7: Diana's total investment returns
# Options: A=$941,447 ... I=$941,455
q7 = match_mc(diana_inv_returns_total, 941447)
print(f"Q7: Diana total inv returns = ${diana_inv_returns_total:,.2f} -> {q7}")

# Q8: Two largest final closing balances
final_bals = {
    'Alice': alice_balances[2044],
    'Bob': bob_balances[2057],
    'Charlie': charlie_balances[2039],
    'Diana': diana_balances[2060]
}
sorted_bals = sorted(final_bals.items(), key=lambda x: x[1], reverse=True)
print(f"Q8: Final balances (sorted): {[(n, f'${v:,.0f}') for n,v in sorted_bals]}")
largest, second = sorted_bals[0][0], sorted_bals[1][0]

# Map to answer options
q8_map = {
    ('Alice', 'Bob'): 'A', ('Alice', 'Charlie'): 'B', ('Alice', 'Diana'): 'C',
    ('Bob', 'Charlie'): 'D', ('Bob', 'Diana'): 'E',
    ('Charlie', 'Bob'): 'F', ('Charlie', 'Diana'): 'G',
    ('Diana', 'Bob'): 'H', ('Diana', 'Charlie'): 'I'
}
q8 = q8_map.get((largest, second), 'UNKNOWN')
print(f"Q8: {largest} largest, {second} second -> {q8}")

# Q9: Sum of all four balances at 31 Dec 2039
total_2039 = alice_balances[2039] + bob_balances[2039] + charlie_balances[2039] + diana_balances[2039]
q9 = round(total_2039)
print(f"Q9: Total at 31 Dec 2039 = ${total_2039:,.2f} -> {q9}")
print(f"    Alice: ${alice_balances[2039]:,.2f}")
print(f"    Bob: ${bob_balances[2039]:,.2f}")
print(f"    Charlie: ${charlie_balances[2039]:,.2f}")
print(f"    Diana: ${diana_balances[2039]:,.2f}")

# Q10: Year Charlie's balance first exceeds $1,200,000 (indefinite)
q10 = charlie_first_1200k
print(f"Q10: Charlie first > $1.2M in {q10}")

In [ ]:
# ============================================================
# SET FINAL ANSWERS
# ============================================================

answers = {
    "question1": q1,
    "question2": q2,
    "question3": q3,
    "question4": q4,
    "question5": q5,
    "question6": q6,
    "question7": q7,
    "question8": q8,
    "question9": q9,
    "question10": q10,
}

print("\nFinal Answers:")
for k, v in answers.items():
    print(f"  {k}: {v}")